# Prerequisite 04 — LFW aligned crop materialization

LFW manifest를 공통 ArcFace 112×112 RGB crop으로 정렬합니다. 이 결과는 모든 Step 2 checkpoint가 공유하며, 검출 실패를 center crop으로 대체하지 않습니다.

**순서:** 이 노트북은 dataset 00 이후, model/embedding 및 Grad-CAM 노트북 이전에 한 번 실행합니다.

In [1]:
from pathlib import Path
import sys
import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("research와 configs가 있는 프로젝트 루트를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

In [2]:
DATA_FRACTION = 1.0
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
OVERWRITE = True
SEED = 42

if DATA_FRACTION != 1.0:
    raise ValueError("공통 aligned crop은 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [3]:
import pandas as pd

from research.preprocessing import materialize_aligned_crops

SOURCE_MANIFEST_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["manifest_path"]
OUTPUT_DIR = PROJECT_ROOT / CONFIG["aligned_crops"]["bundle_dir"]
ALIGNMENT_PROVIDERS = tuple(CONFIG["aligned_crops"]["providers"])
REQUIRED_PRIMARY_PROVIDER = CONFIG["aligned_crops"]["required_primary_provider"]
if not ALIGNMENT_PROVIDERS or ALIGNMENT_PROVIDERS[0] != REQUIRED_PRIMARY_PROVIDER:
    raise ValueError("aligned-crop primary provider 설정이 일치하지 않습니다.")

In [4]:
if EXECUTE_STAGE:
    if not SOURCE_MANIFEST_PATH.is_file():
        raise FileNotFoundError(
            "먼저 notebooks/lfw/00_data_preparation/00_data_preparation.ipynb를 "
            f"실행하세요: {SOURCE_MANIFEST_PATH}"
        )
    source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH)
    result = materialize_aligned_crops(
        source_manifest,
        project_root=PROJECT_ROOT,
        output_dir=OUTPUT_DIR,
        dataset_id="lfw",
        providers=ALIGNMENT_PROVIDERS,
        overwrite=OVERWRITE,
    )
    summary = {
        "output_dir": str(result.output_dir),
        **result.bundle_manifest["counts"],
        "detector": result.bundle_manifest["detector"],
        "array_contract": result.bundle_manifest["array_contract"],
    }
else:
    summary = {"status": "not_executed"}
summary

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'cudnn_conv_algo_search': 'EXHAUSTIVE', 'device_id': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'has_user_compute_stream': '0', 'gpu_external_alloc': '0', 'enable_cuda_graph': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0'}, 'CPUExecutionProvider': {}}
model ignore: C:\Users\Administrator/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'cudnn_conv_algo_search': 'EXHAUSTIVE', 'device_id': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'has_user_compute_stream': '0

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


{'output_dir': 'C:\\ronbun\\data\\interim\\common\\aligned_112',
 'source': 13233,
 'aligned': 13195,
 'failed': 38,
 'detector': {'name': 'buffalo_l',
  'detection_size': [640, 640],
  'allowed_modules': ['detection'],
  'requested_providers': ['CUDAExecutionProvider', 'CPUExecutionProvider'],
  'providers': ['CUDAExecutionProvider', 'CPUExecutionProvider']},
 'array_contract': {'shape': [13195, 112, 112, 3],
  'dtype': 'uint8',
  'layout': 'nhwc',
  'color_order': 'rgb',
  'image_size': [112, 112]}}

## 저장 계약

- `aligned_faces.npy`: 모델 입력용 binary array
- `aligned_index.csv`: 사람이 확인 가능한 sample↔array index
- `failed_samples.csv`: 실패 사유 전수
- `bundle_manifest.json`, `_SUCCESS`: hash·shape·완료 상태

`OVERWRITE=True`는 완성된 대체 bundle을 먼저 staging한 뒤 canonical 폴더 하나만 교체합니다. 이를 사용한 완료 run은 기록된 입력 hash와 함께 계속 불변입니다.